# 🏀 NBA Data Update Demo

**Quick demo to update NBA player data through the current date (November 12, 2025)**

This notebook shows how to:
1. Collect data for specific players and seasons
2. Update existing data files with new games
3. Verify data coverage through the current date

## Setup

In [1]:
import sys
import os
from datetime import datetime
import pandas as pd
import time

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from nba_player_props.data_collector.data_collector import DataCollector

print(f"✅ Setup complete!")
print(f"📅 Today: {datetime.now().strftime('%Y-%m-%d')}")

✅ Setup complete!
📅 Today: 2025-11-12


## 📥 Collect Data for Demo Players

Let's collect 2024-25 season data for a few key players to demonstrate the update process.

In [2]:
# Initialize collector with reasonable delay
collector = DataCollector(rate_limit_delay=2.0)

# Define players for demo (using flexible_collect.py approach)
demo_players = [
    "LeBron James",
    "Stephen Curry", 
    "Nikola Jokic",
]

current_season = "2024-25"

print(f"🏀 Collecting {current_season} data for {len(demo_players)} players...")
print(f"⏰ Started: {datetime.now().strftime('%H:%M:%S')}\n")

🏀 Collecting 2024-25 data for 3 players...
⏰ Started: 20:11:32



In [3]:
# Collect data for each player (mimicking flexible_collect.py logic)
all_data = []
successful_players = []
failed_players = []

for i, player_name in enumerate(demo_players, 1):
    print(f"📍 {i}/{len(demo_players)}: {player_name}")
    
    try:
        player_data = collector.get_player_data_by_name(
            player_name=player_name,
            seasons=[current_season],
            include_playoffs=False  # Regular season only for now
        )
        
        if not player_data.empty:
            all_data.append(player_data)
            successful_players.append(player_name)
            print(f"   ✅ Collected {len(player_data)} games")
            
            # Show latest game
            if 'GAME_DATE' in player_data.columns:
                latest_game = player_data['GAME_DATE'].max()
                print(f"   📅 Latest game: {latest_game}")
        else:
            failed_players.append(player_name)
            print(f"   ⚠️  No data found")
    
    except Exception as e:
        failed_players.append(player_name)
        print(f"   ❌ Error: {str(e)}")
    
    # Delay between players (like flexible_collect.py)
    if i < len(demo_players):
        print(f"   ⏳ Waiting 3 seconds...")
        time.sleep(3)
    print()

print(f"⏰ Completed: {datetime.now().strftime('%H:%M:%S')}")
print(f"✅ Successful: {len(successful_players)}")
print(f"❌ Failed: {len(failed_players)}")

📍 1/3: LeBron James
   ✅ Collected 70 games
   📅 Latest game: 2025-04-11 00:00:00
   ⏳ Waiting 3 seconds...

📍 2/3: Stephen Curry
   ✅ Collected 70 games
   📅 Latest game: 2025-04-13 00:00:00
   ⏳ Waiting 3 seconds...


KeyboardInterrupt: 

## 🔄 Process & Combine Data

Combine collected data and handle duplicates (following flexible_collect.py approach).

In [ ]:
if all_data:
    # Combine all player data
    combined_df = pd.concat(all_data, ignore_index=True)
    
    # Remove duplicates (like flexible_collect.py)
    initial_count = len(combined_df)
    combined_df = combined_df.drop_duplicates(subset=['PLAYER_ID', 'Game_ID'])
    final_count = len(combined_df)
    
    if initial_count != final_count:
        print(f"🔧 Removed {initial_count - final_count} duplicate games")
    
    # Ensure GAME_DATE is properly formatted
    combined_df['GAME_DATE'] = pd.to_datetime(combined_df['GAME_DATE'], errors='coerce')
    
    print(f"\n📊 Combined Dataset Summary:")
    print(f"   Total games: {len(combined_df):,}")
    print(f"   Unique players: {combined_df['PLAYER_ID'].nunique()}")
    print(f"   Date range: {combined_df['GAME_DATE'].min()} to {combined_df['GAME_DATE'].max()}")
    print(f"   Seasons: {combined_df['SEASON'].unique().tolist()}")
    
else:
    print("❌ No data collected")
    combined_df = pd.DataFrame()

## 📊 Explore the Data

Let's look at what we collected.

In [ ]:
if not combined_df.empty:
    # Show sample of the data
    print("📋 Sample of collected data:")
    display(combined_df[['PLAYER_NAME', 'GAME_DATE', 'MATCHUP', 'PTS', 'REB', 'AST']].head(10))
    
    # Show most recent games
    print("\n🆕 Most Recent Games:")
    recent = combined_df.nlargest(5, 'GAME_DATE')[['PLAYER_NAME', 'GAME_DATE', 'MATCHUP', 'PTS', 'REB', 'AST']]
    display(recent)

In [ ]:
if not combined_df.empty:
    # Player statistics for the season
    print("📈 Player Season Averages:")
    
    stats = combined_df.groupby('PLAYER_NAME').agg({
        'PTS': 'mean',
        'REB': 'mean',
        'AST': 'mean',
        'Game_ID': 'count'
    }).round(1)
    
    stats.columns = ['PPG', 'RPG', 'APG', 'Games']
    stats = stats.sort_values('PPG', ascending=False)
    
    display(stats)

## 💾 Save or Update Data Files

Following the `flexible_collect.py` pattern to save/append data.

In [ ]:
if not combined_df.empty:
    # Set up output directory and file (following flexible_collect.py pattern)
    output_dir = "../data/player_box_scores"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = f"{output_dir}/nba_player_games_{current_season}.csv"
    
    # Check if file exists (append logic from flexible_collect.py)
    if os.path.exists(output_file):
        print(f"📄 Existing file found for {current_season}")
        existing_df = pd.read_csv(output_file)
        print(f"   Existing games: {len(existing_df):,}")
        
        # Combine with existing data
        full_df = pd.concat([existing_df, combined_df], ignore_index=True)
        
        # Remove duplicates
        pre_dedup = len(full_df)
        full_df = full_df.drop_duplicates(subset=['PLAYER_ID', 'Game_ID'])
        post_dedup = len(full_df)
        
        if pre_dedup != post_dedup:
            print(f"   🔧 Removed {pre_dedup - post_dedup} duplicates after merge")
        
        new_games = len(full_df) - len(existing_df)
        print(f"   📈 Added {new_games} new games")
        
    else:
        print(f"📄 Creating new file for {current_season}")
        full_df = combined_df
    
    # Save the file
    full_df.to_csv(output_file, index=False)
    file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
    
    print(f"\n🎉 SUCCESS!")
    print(f"   💾 File: {output_file}")
    print(f"   📊 Total games: {len(full_df):,}")
    print(f"   👥 Players: {full_df['PLAYER_ID'].nunique()}")
    print(f"   💿 Size: {file_size_mb:.2f} MB")
    
    # Show date range
    if 'GAME_DATE' in full_df.columns:
        full_df['GAME_DATE'] = pd.to_datetime(full_df['GAME_DATE'], errors='coerce')
        min_date = full_df['GAME_DATE'].min().strftime('%Y-%m-%d')
        max_date = full_df['GAME_DATE'].max().strftime('%Y-%m-%d')
        print(f"   📅 Date range: {min_date} to {max_date}")

else:
    print("❌ No data to save")

---

## 🚀 Next Steps

### For Full Updates

To update all players, you can use the command-line scripts:

**Option 1: Update specific players**
```bash
# Using flexible_collect.py for targeted updates
python flexible_collect.py --players "LeBron James,Stephen Curry,Nikola Jokic" --seasons 2024-25
```

**Option 2: Update from predefined lists**
```bash
# Update the "original_30" player list
python flexible_collect.py --players original_30 --seasons 2024-25

# Update the "second_wave_15" additions
python flexible_collect.py --players second_wave_15 --seasons 2023-24 2024-25
```

**Option 3: Update multiple seasons**
```bash
# Collect data for a range of seasons
python flexible_collect.py --players original_30 --start 2020 --end 2024
```

### Available Player Lists
- `original_30` - Core superstars and key players
- `second_wave_15` - Next tier of high-value players  
- `third_wave_15` - Additional quality players for prop betting
- `legends_2000s` - Historical stars from 2000s era

See `flexible_collect.py` for the full list of players in each category!